In [0]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from pyspark.sql import SparkSession

silver_df = spark.table("ddca26online23.default.silver_player_history")

silver_df = silver_df.withColumn(
    "target",
    F.when(F.col("outcome") == "Win", 1).otherwise(0)
)

w = Window.partitionBy("player_id").orderBy("match_id").rowsBetween(-5, -1)

feature_df = silver_df.withColumn(
    "avg_team_score_last5", F.avg("team_score").over(w)
).withColumn(
    "avg_opp_score_last5", F.avg("opponent_score").over(w)
).withColumn(
    "winrate_last5", F.avg("target").over(w)
)


ml_df = feature_df.select(
    "player_id",
    "avg_team_score_last5",
    "avg_opp_score_last5",
    "winrate_last5",
    "target"
).toPandas()

print(ml_df.head())
ml_df = ml_df.dropna()

X = ml_df[[
    "avg_team_score_last5",
    "avg_opp_score_last5",
    "winrate_last5"
]]
y = ml_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.autolog()

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

ml_df['pred_prob'] = model.predict_proba(X)[:, 1]

spark_df = SparkSession.builder.getOrCreate().createDataFrame(ml_df[['player_id','pred_prob']])
spark_df.write.format("delta").mode("overwrite").saveAsTable("ddca26online23.default.player_pred_probs")